# Fine-turing FinBERT

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_scheduler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.utils.data as data
import torch.optim as optim
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Prepare dataset

In [ ]:
df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/data_coindesk_base.csv')
df = df[['ID', 'BODY', 'SENTIMENT', 'PUBLISHED_ON']]

df.loc[df['SENTIMENT'] == 'POSITIVE', ['SENTIMENT']] = 0
df.loc[df['SENTIMENT'] == 'NEGATIVE', ['SENTIMENT']] = 1
df.loc[df['SENTIMENT'] == 'NEUTRAL', ['SENTIMENT']] = 2

df = df.sample(frac=1, random_state=52).reset_index()
df.dropna(inplace=True)
df = df[:50_000]

X = df['BODY']
y = df['SENTIMENT']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

## Download pre-trained model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=3).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model.config.id2label

{0: 'positive', 1: 'negative', 2: 'neutral'}

## Tokenizer & Model architecture

In [ ]:
tokenizer

BertTokenizerFast(name_or_path='ProsusAI/finbert', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
tokenizer.model_input_names

['input_ids', 'token_type_ids', 'attention_mask']

tokenizer.model_input_names — это список строк, которые указывают на имена параметров, которые модель ожидает получить в качестве входных данных. В данном случае, это имена аргументов, которые вы должны передать токенизатору, чтобы он сгенерировал данные, необходимые для вашей модели.

Для большинства моделей на основе Transformer, включая FinBERT, эти имена обычно включают:

    'input_ids': это последовательность числовых идентификаторов токенов (слов или частей слов) в вашем тексте.
    'attention_mask': это маска, которая указывает, какие токены являются реальными словами, а какие — "паддингом" (заполнителями), чтобы сделать все последовательности одинаковой длины. Это помогает модели игнорировать токены-заполнители.
    'token_type_ids': это идентификаторы типов токенов, которые используются для различения разных сегментов в одном входе (например, если вы подаете два предложения в модель для задачи определения логического следования) Эти идентификаторы используются, чтобы различать сегменты, когда вы подаете более одного предложения как единый вход в модель (например, для задач "Вопрос-Ответ", где есть вопрос и контекст, или "Вывод естественного языка", где есть два предложения). В таком случае первое предложение получает 0, а второе — 1.


In [ ]:
tokenizer(['hello, world!', 'what? how are you? i am going to play'])

{'input_ids': [[101, 7592, 1010, 2088, 999, 102], [101, 2054, 1029, 2129, 2024, 2017, 1029, 1045, 2572, 2183, 2000, 2377, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

### Example for token_type_ids

In [ ]:
tokenizer(['hello, world!', ['what? how are you?',  'i am going to play']])

{'input_ids': [[101, 7592, 1010, 2088, 999, 102], [101, 2054, 1029, 2129, 2024, 2017, 1029, 102, 1045, 2572, 2183, 2000, 2377, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

### Model

In [ ]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
next(iter(model.bert.parameters()))

Parameter containing:
tensor([[-0.0105, -0.0592, -0.0243,  ..., -0.0236, -0.0433, -0.0114],
        [-0.0113, -0.0586, -0.0287,  ..., -0.0188, -0.0406, -0.0116],
        [-0.0191, -0.0610, -0.0313,  ..., -0.0171, -0.0444, -0.0033],
        ...,
        [-0.0229, -0.0575, -0.0117,  ..., -0.0075, -0.0162, -0.0263],
        [-0.0453, -0.0531,  0.0003,  ...,  0.0135, -0.0201, -0.0112],
        [ 0.0018, -0.0752, -0.0137,  ..., -0.0135, -0.0520,  0.0743]],
       device='cuda:0', requires_grad=True)

## Dataset

In [ ]:
class ArticlesDataset:
    def __init__(self, articles, targets, tokenizer):
        self.articles = list(articles.values)
        self.tokenizer = tokenizer
        self.articles_ids = self.tokenize_func(self.articles)
        self.targets = targets

    def __getitem__(self, indx):
        art_ids = self.articles_ids['input_ids'][indx]
        art_types_ids = self.articles_ids['token_type_ids'][indx] # zeros
        art_att_mask = self.articles_ids['attention_mask'][indx]

        target = self.targets[indx]

        result_dict = {
            'article_ids': art_ids,
            'article_type_ids': art_types_ids,
            'article_attention_mask': art_att_mask,
            'target': target}

        return result_dict

    def __len__(self):
        return len(self.articles)

    def tokenize_func(self, arts):
        return self.tokenizer(arts, truncation=True,
                              padding='max_length', max_length=512,
                              padding_side='right')

## Collate batch function

In [ ]:
def collate_batch(batch, device=device):
    arts_ids, att_masks, arts_type_ids, targets = [], [], [], []

    for dct in batch:
        arts_ids.append(dct['article_ids'])
        arts_type_ids.append(dct['article_type_ids'])
        att_masks.append(dct['article_attention_mask'])
        targets.append(dct['target'])

    articles_ids = torch.LongTensor(arts_ids).to(device)
    articles_types_ids = torch.LongTensor(arts_type_ids).to(device)
    articles_att_masks = torch.LongTensor(att_masks).to(device)
    targets = torch.LongTensor(targets).to(device)

    return {
        'article_ids': articles_ids,
        'article_type_ids': articles_types_ids,
        'article_attention_mask': articles_att_masks,
        'target': targets}

## Datasets & Dataloaders

In [ ]:
train_ds = ArticlesDataset(X_train, y_train, tokenizer)
val_ds = ArticlesDataset(X_val, y_val, tokenizer)
test_ds = ArticlesDataset(X_test, y_test, tokenizer)

train_dl = data.DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_batch)
val_dl = data.DataLoader(val_ds, batch_size=32, shuffle=True, collate_fn=collate_batch)
test_dl = data.DataLoader(test_ds, batch_size=32, shuffle=True, collate_fn=collate_batch)

### Explain model outputs

In [ ]:
sample_input_ids = torch.tensor([[101, 7592, 1010, 2088, 999, 102]]).to(device)
sample_attention_mask = torch.tensor([[1, 1, 1, 1, 1, 1]]).to(device)
sample_token_type_ids = torch.tensor([[0, 0, 0, 0, 0, 0]]).to(device)

model_output = model(
    input_ids=sample_input_ids,
    attention_mask=sample_attention_mask,
    token_type_ids=sample_token_type_ids
)

print(model_output)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.2528, -1.2433,  1.9860]], device='cuda:0',
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


## Evaluation

In [ ]:
def evaluate(model, dl):
    model.eval()

    loss_fn = nn.CrossEntropyLoss()

    losses = []
    preds = []
    targets = []

    with torch.no_grad():
        for batch in tqdm(dl):
            pred = model(
                input_ids=batch['article_ids'],
                token_type_ids=batch['article_type_ids'],
                attention_mask=batch['article_attention_mask'],
                labels=batch['target']
            )
            loss = loss_fn(pred.logits, batch['target'])

            losses.append(loss.item())
            preds.append(pred.logits.argmax(dim=-1))
            targets.append(batch['target'])

    preds = torch.cat(preds, dim=0).to(device)
    targets = torch.cat(targets).to(device) # default dim=0

    mean_loss = np.mean(losses)
    accuracy = (targets == preds).float().mean().item()

    model.train()

    return mean_loss, accuracy

In [ ]:
start_eval = list(evaluate(model, val_dl))
print(f'Mean_loss: {start_eval[0]}', f'Accuracy: {start_eval[1]}')

  0%|          | 0/132 [00:00<?, ?it/s]

Mean_loss: 1.1072243890075972 Accuracy: 0.5525999665260315


Итак, начальные результаты правильной классификации новостей 55%

посмотрим, что произойдет после Fine-turing

## Fine-turing

In [ ]:
EPOCHS = 10
LR = 1e-5

optimizer = optim.AdamW(params=model.parameters(), lr=LR)
loss_func = nn.CrossEntropyLoss()
num_training_steps = len(train_dl) * EPOCHS

scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps)

model_path = '/content/gdrive/MyDrive/Colab Notebooks/course/notebooks/transformers/model_finbert_v0'
progress_bar = tqdm(range(num_training_steps))

count_early = 0
best_loss_val = float('inf')
model.train()

accuracies = {'train': [], 'val': []}
losses = {'train': [], 'val': []}

for epoch in range(EPOCHS):
    curr_loss = []
    preds = []
    targets = []

    for i, batch in enumerate(tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS}')):
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch['article_ids'],
            attention_mask=batch['article_attention_mask'],
            token_type_ids=batch['article_type_ids'],
            labels=batch['target'])
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        progress_bar.update(1)

        curr_loss.append(loss.item())
        preds.append(outputs.logits.argmax(dim=-1))
        targets.append(batch['target'])


    preds = torch.cat(preds)
    targets = torch.cat(targets)

    epoch_acc = (preds == targets).float().mean().item()
    epoch_loss = np.mean(curr_loss)

    # EVALUATION
    mean_loss_val, acc_val = evaluate(model, val_dl)

    losses['train'].append(epoch_loss)
    accuracies['train'].append(epoch_acc)

    losses['val'].append(mean_loss_val)
    accuracies['val'].append(acc_val)

    print(f'Loss train/val in this epoch: {epoch_loss}/{mean_loss_val}')
    print(f'Accuracy train/val in this epoch: {epoch_acc}/{acc_val}')

    if mean_loss_val < best_loss_val:
        best_loss_val = mean_loss_val
        count_early = 0

        st = model.state_dict()
        torch.save(st, model_path)
    else:
        count_early += 1

    if count_early > 1:
        print('Early stopping!')
        break


  0%|          | 0/10530 [00:00<?, ?it/s]

Epoch 1/10:   0%|          | 0/1053 [00:00<?, ?it/s]

  0%|          | 0/132 [00:00<?, ?it/s]

Loss train/val in this epoch: 0.5536978230347321/0.48287637892997626
Accuracy train/val in this epoch: 0.7651249766349792/0.7963999509811401


Epoch 2/10:   0%|          | 0/1053 [00:00<?, ?it/s]

...

  0%|          | 0/132 [00:00<?, ?it/s]

Loss train/val in this epoch: 0.32325207147701285/0.4624713917799068
Accuracy train/val in this epoch: 0.8717749714851379/0.819599986076355


Epoch 4/10:   0%|          | 0/1053 [00:00<?, ?it/s]

  0%|          | 0/132 [00:00<?, ?it/s]

Loss train/val in this epoch: 0.24666700665003213/0.5270429113597581
Accuracy train/val in this epoch: 0.9062249660491943/0.8149999976158142
Early stopping!


In [ ]:
torch.cuda.empty_cache()